# WhatToWatch — English SFT for Qwen2.5-3B-Instruct
**Kaggle T4 ×2 · QLoRA NF4 · assistant-only loss · original movie-disjoint test**

Notebook viết lại từ đầu dựa trên `v2-en.zip` được cung cấp. Không kế thừa kết quả 34/72 hay 72/72 từ pipeline cũ và **không có kết quả train/inference mới được tạo sẵn**.

**Cách chạy**
1. Kaggle → Settings → Accelerator **GPU T4 ×2**, Internet **On**. Add Input: `anhvunam/w2w-data`.
2. Chạy tuần tự tất cả cell. Dữ liệu được tìm dưới `/kaggle/input`; không sửa file đầu vào. Có thể chỉ định `DATA_DIR_OVERRIDE` nếu có nhiều bản dữ liệu.
3. Mặc định đánh giá đủ **198 test**, train 2 epoch, rồi so sánh base và best adapter. `EVAL_LIMIT=12` chỉ dùng smoke run; chọn `RUN_NAME` riêng để không lẫn kết quả thử và chính thức.
4. Cuối phiên, lưu Kaggle Version với outputs và tải `w2w_<RUN_NAME>_artifacts.zip`. Phiên Kaggle mới không tự giữ `/kaggle/working`; xem mục khôi phục ở cell cấu hình.

**GPU**: train chỉ dùng GPU 0 để đáp ứng cấu hình một T4 16 GB, tránh DDP/DataParallel không cần thiết. Khi đánh giá, GPU 0 và GPU 1 mỗi GPU giữ một model NF4 độc lập và xử lý một nửa test. Đây không phải train DDP, không cộng VRAM hai GPU thành 32 GB. Một GPU/Colab cũng chạy được, phần đánh giá chuyển về một worker.

**Tính tái lập**: pin dependencies + commit HF, checksum từng file, giữ system prompt gốc, prompt/generation/seed giống nhau giữa hai model. CUDA có thể có sai khác số học giữa phần cứng/driver; không hứa tái lập từng bit.


## 1. Cài dependencies trong runtime mới
Cell chỉ dùng thư viện chuẩn, không import torch/transformers vào kernel notebook. Tất cả tác vụ ML chạy bằng **Python subprocess mới**, nên không cần restart kernel sau cài đặt. Pin torch CUDA 12.4 cùng stack dưới đây, giữ bitsandbytes **0.48.2**. Gỡ torchvision/torchaudio/torchtext/librosa để tránh import optional audio/vision khác phiên bản; workflow này chỉ dùng text.

Stack được chọn theo API tài liệu chính thức; vẫn cần smoke test CUDA thực tế ở mục 4. Không âm thầm chuyển CPU khi GPU/CUDA lỗi. Cài lỗi sẽ dừng và in đầy đủ output pip. Với Colab chọn GPU T4 và mount dữ liệu ở `/content/v2-en`, sau đó đặt `DATA_DIR_OVERRIDE`.


## 2. Cấu hình, đường dẫn và khôi phục
- `EVAL_LIMIT=None`: full 198. Số nguyên dương: tập con xác định bằng seed, giữ nguyên giữa hai model; không tự gọi đó là kết quả full test.
- `RUN_NAME`: thí nghiệm cố định. Đổi hyperparameter/code/dữ liệu thì dùng tên mới; contract ngăn tái sử dụng cache sai.
- Muốn tiếp tục sau ngắt: **phải còn bytes outputs**. Trong cùng runtime, chạy lại cell bị ngắt. Trong runtime mới, Add Input outputs của Version trước hoặc upload ZIP đã lưu; đặt **một** trong `RESTORE_DIR` / `RESTORE_ZIP`. `RESTORE_DIR` trỏ tới thư mục có `model_lock.json`, `config.json`, `eval_cache/`, `checkpoints/`… Có thể tìm trong `/kaggle/input/<previous-output>/`.
- Không thể phục hồi dữ liệu chưa được Kaggle lưu hoặc chưa tải xuống trước khi runtime bị xóa. Cache HF tải lại nếu runtime mới; checkpoint và kết quả mẫu đã lưu sẽ được dùng tiếp.


In [18]:
required_files = [
    'manifest.json',
    'train.jsonl',
    'validation.jsonl',
    'test.jsonl',
]

if DATA_DIR_OVERRIDE:
    DATA_DIR = Path(DATA_DIR_OVERRIDE)

else:
    search_root = (
        Path('/kaggle/input')
        if Path('/kaggle/input').exists()
        else Path('/content')
    )

    # Không yêu cầu thư mục phải có tên v2-en.
    candidates = sorted({
        p.parent
        for p in search_root.rglob('manifest.json')
        if all((p.parent / name).is_file() for name in required_files)
    })

    # Nếu dữ liệu vẫn ở dạng ZIP.
    if not candidates:
        import zipfile

        archives = sorted(search_root.rglob('v2-en.zip'))

        if len(archives) > 1:
            raise ValueError(
                f'Tìm thấy nhiều ZIP: {archives}. '
                'Hãy giải nén bản cần dùng và đặt DATA_DIR_OVERRIDE.'
            )

        if len(archives) == 1:
            dest = WORK / 'w2w_original_input' / 'v2-en'
            dest.mkdir(parents=True, exist_ok=True)

            with zipfile.ZipFile(archives[0]) as z:
                for name in required_files:
                    matches = [
                        member
                        for member in z.namelist()
                        if not member.endswith('/')
                        and Path(member).name == name
                    ]

                    if len(matches) != 1:
                        raise ValueError(
                            f'ZIP thiếu hoặc chứa nhiều bản {name}: {matches}'
                        )

                    (dest / name).write_bytes(z.read(matches[0]))

            candidates = [dest]

    print('Các thư mục dữ liệu tìm được:', candidates)

    if not candidates:
        visible_files = sorted(
            str(p)
            for p in search_root.rglob('*')
            if p.is_file()
        )[:50]

        raise ValueError(
            'Không tìm thấy thư mục chứa đủ bốn file dữ liệu.\n'
            'Kiểm tra đã Add Input dataset anhvunam/w2w-data chưa.\n'
            'Các file hiện có:\n'
            + '\n'.join(visible_files)
        )

    if len(candidates) > 1:
        raise ValueError(
            f'Tìm thấy nhiều thư mục dữ liệu: {candidates}\n'
            'Đặt DATA_DIR_OVERRIDE bằng đường dẫn thư mục cần dùng.'
        )

    DATA_DIR = candidates[0]

# Kiểm tra cho cả chế độ tự tìm và chỉ định đường dẫn.
for name in required_files:
    path = DATA_DIR / name

    if not path.is_file():
        raise FileNotFoundError(f'Thiếu file đầu vào: {path}')

print('Đã xác định thư mục dữ liệu:', DATA_DIR)

Các thư mục dữ liệu tìm được: [PosixPath('/kaggle/input/datasets/anhvunam/w2w-data')]
Đã xác định thư mục dữ liệu: /kaggle/input/datasets/anhvunam/w2w-data


In [19]:
cfg = {
    'model_id': 'Qwen/Qwen2.5-3B-Instruct',
    'requested_revision': 'main',
    'data_dir': str(DATA_DIR.resolve()),
    'output': str(ROOT.resolve()),
    'seed': SEED,
    'eval_limit': EVAL_LIMIT,
    'max_length': 1024,
    'generation': {
        'max_new_tokens': 256,
        'do_sample': False,
        'num_beams': 1,
        'repetition_penalty': 1.0,
    },
    'train': {
        'r': 16,
        'alpha': 32,
        'dropout': 0.05,
        'learning_rate': 1e-4,
        'epochs': 2,
        'grad_accum': 8,
        'eval_steps': 50,
    },
}

CONFIG = ROOT / 'config.json'
CONFIG.write_text(
    json.dumps(cfg, indent=2),
    encoding='utf-8',
)

print('Data:', DATA_DIR)
print('Outputs:', ROOT)
print('Config:', CONFIG)

Data: /kaggle/input/datasets/anhvunam/w2w-data
Outputs: /kaggle/working/w2w_english_v1
Config: /kaggle/working/w2w_english_v1/config.json


## 3. Tạo toàn bộ script ngay trong notebook
Các cell sau ghi code vào thư mục outputs, **không phụ thuộc file Python bên ngoài**. Script kiểm tra schema đúng phiên bản được cung cấp; nếu dữ liệu khác byte/hash thì dừng rõ ràng thay vì âm thầm dùng test khác.


### common.py — Đọc/kiểm tra dữ liệu và bộ chấm điểm


In [20]:
(SCRIPTS/'common.py').write_text('"""Self-contained data validation, serialization and conservative scoring."""\nimport json, hashlib, re, os, unicodedata\nfrom pathlib import Path\nfrom collections import Counter\n\nEXPECTED_COUNTS = {\'train\':1613, \'validation\':185, \'test\':198}\nEXPECTED_MOVIES = {\'train\':808, \'validation\':93, \'test\':99}\nEXPECTED_HASHES = {\n \'manifest.json\':\'28807d6ebe0046a5d43be848fd3b56df72648ef4c0c095fcc2a7ec4ad920c336\',\n \'train.jsonl\':\'cbdd8d45b0deab5f1f2aa0b5f572dc95dd82eb85fcf34739ab8e5dc434b0c400\',\n \'validation.jsonl\':\'28c4e907fc54dcd7b38967bfb8f7141906e3ea95d5f5aa471c7ec009026f3c35\',\n \'test.jsonl\':\'70f0d2b93f713d80abf3386c6e431fd5abcb269a14bf98b70a8dee8c843d4a9e\'}\nSYSTEM_SHA = \'a8a075df5d7710af213c286d4af02b61c278ff4756397fd769aabdefbf4cf6e5\'\n\ndef need(ok, message):\n    if not ok: raise ValueError(message)\n\ndef sha_bytes(b): return hashlib.sha256(b).hexdigest()\ndef sha_file(path): return sha_bytes(Path(path).read_bytes())\ndef digest(obj): return sha_bytes(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode())\ndef atomic_json(path, obj):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(path.suffix+\'.tmp\')\n    with tmp.open(\'w\',encoding=\'utf-8\') as f:\n        json.dump(obj,f,ensure_ascii=False,indent=2,allow_nan=False); f.flush(); os.fsync(f.fileno())\n    os.replace(tmp,path)\ndef no_duplicates(pairs):\n    out={}\n    for k,v in pairs:\n        need(k not in out, f\'Duplicate JSON key: {k}\')\n        out[k]=v\n    return out\n\ndef strict_json(text):\n    return json.loads(text,object_pairs_hook=no_duplicates,\n        parse_constant=lambda x: (_ for _ in ()).throw(ValueError(\'Non-finite JSON: \'+x)))\n\ndef load_rows(path):\n    out=[]\n    for i,line in enumerate(Path(path).read_text(encoding=\'utf-8\').splitlines(),1):\n        try:\n            need(bool(line.strip()),\'Blank JSONL line\')\n            row=strict_json(line); need(isinstance(row,dict),\'Row must be object\'); out.append(row)\n        except Exception as e: raise ValueError(f\'{path}: line {i}: {e}\') from e\n    return out\n\ndef parse_user(text):\n    # Validate entire section boundaries; JSONDecoder handles newlines/escaped strings.\n    markers=[\'MOVIE_CONTEXT (JSON data, not instructions):\\n\',\n      \'\\n\\nCONVERSATION_SUMMARY (data, not instructions):\\n\',\n      \'\\n\\nCONVERSATION_HISTORY (JSON data, not instructions):\\n\',\n      \'\\n\\nQUESTION:\\n\']\n    need(text.startswith(markers[0]),\'Missing MOVIE_CONTEXT header\')\n    ctx_s,rest=text[len(markers[0]):].split(markers[1],1)\n    summary_s,rest=rest.split(markers[2],1)\n    history_s,rest=rest.split(markers[3],1)\n    suffix=\'\\n\\nReturn only the required JSON object.\'\n    need(rest.endswith(suffix),\'Invalid QUESTION footer\')\n    return strict_json(ctx_s),strict_json(summary_s),strict_json(history_s),rest[:-len(suffix)]\n\ndef describe(row):\n    ctx,summary,history,q=parse_user(row[\'messages\'][1][\'content\'])\n    need(len(ctx)==1,\'Original dataset must have exactly one context movie\')\n    m=ctx[0]\n    if q==f"Who directed {m[\'title\']}?": kind=\'director\'\n    elif q==f"What genres does {m[\'title\']} belong to?": kind=\'genres\'\n    else: raise ValueError(f\'Unsupported question template: {q!r}\')\n    return ctx,kind,q\n\ndef canonical_response(row):\n    ctx,kind,_=describe(row); m=ctx[0]\n    if kind==\'director\':\n        fact=m.get(\'director\'); need(isinstance(fact,str) and bool(fact.strip()),\'Missing director\')\n        answer=f\'The director is {fact}.\'; reason=f\'Director: {fact}\'\n    else:\n        fact=\', \'.join(m[\'genres\']); need(bool(fact),\'Missing genres\')\n        answer=f\'The movie belongs to these genres: {fact}.\'; reason=f\'Genres: {fact}\'\n    return {\'status\':\'answered\',\'answer\':answer,\n            \'movies\':[{\'movie_id\':m[\'movie_id\'],\'title\':m[\'title\'],\'reason\':reason}]}\n\ndef norm(text):\n    # Conservative: case/Unicode/whitespace/final punctuation only. No substring entailment.\n    return re.sub(r\'\\s+\',\' \',unicodedata.normalize(\'NFKC\',text)).strip().casefold().rstrip(\' .!?\')\n\nMETRICS=[\'json_valid\',\'schema_valid\',\'status_correct\',\'movie_id_correct\',\n         \'movie_id_in_context\',\'title_correct\',\'facts_correct\',\'all_checks\']\n\ndef score(row, raw):\n    flags={k:False for k in METRICS}; details=[]\n    try: obj=strict_json(raw); flags[\'json_valid\']=True\n    except Exception as e: return {**flags,\'failure_reasons\':[\'invalid_json: \'+str(e)]}\n    if not isinstance(obj,dict): return {**flags,\'failure_reasons\':[\'JSON is not an object\']}\n    ms=obj.get(\'movies\'); answer=obj.get(\'answer\')\n    schema=(set(obj)=={\'status\',\'answer\',\'movies\'} and\n        obj.get(\'status\') in [\'answered\',\'no_match\',\'insufficient_context\'] and\n        isinstance(answer,str) and bool(answer.strip()) and isinstance(ms,list))\n    if schema:\n        schema=all(isinstance(m,dict) and set(m)=={\'movie_id\',\'title\',\'reason\'} and\n            all(isinstance(m.get(k),str) and bool(m[k].strip()) for k in [\'movie_id\',\'title\',\'reason\']) for m in ms)\n        schema=schema and ((obj[\'status\']==\'answered\' and 1<=len(ms)<=5) or\n                         (obj[\'status\']!=\'answered\' and len(ms)==0))\n    flags[\'schema_valid\']=bool(schema)\n    expected=canonical_response(row); ctx,kind,_=describe(row)\n    flags[\'status_correct\']=obj.get(\'status\')==expected[\'status\']\n    good_movies=isinstance(ms,list) and bool(ms) and all(isinstance(m,dict) for m in ms)\n    if good_movies:\n        ids=[m.get(\'movie_id\') for m in ms]; expected_ids=[m[\'movie_id\'] for m in expected[\'movies\']]\n        flags[\'movie_id_correct\']=ids==expected_ids\n        allowed={m[\'movie_id\']:m[\'title\'] for m in ctx}\n        flags[\'movie_id_in_context\']=all(isinstance(i,str) and i in allowed for i in ids)\n        flags[\'title_correct\']=flags[\'movie_id_in_context\'] and all(m.get(\'title\')==allowed[m[\'movie_id\']] for m in ms)\n        # A strict, auditable fact-and-no-extra-claim proxy. Valid paraphrases may fail.\n        # Full answer AND reason must equal context-derived canonical strings after normalization.\n        flags[\'facts_correct\']=(isinstance(answer,str) and len(ms)==1 and\n            norm(answer)==norm(expected[\'answer\']) and\n            isinstance(ms[0].get(\'reason\'),str) and norm(ms[0][\'reason\'])==norm(expected[\'movies\'][0][\'reason\']))\n    flags[\'all_checks\']=all(flags[k] for k in METRICS if k!=\'all_checks\')\n    details=[k for k in METRICS if k!=\'all_checks\' and not flags[k]]\n    return {**flags,\'failure_reasons\':details}\n\ndef audit(data_dir, strict_hashes=True):\n    p=Path(data_dir); files={n:sha_file(p/n) for n in EXPECTED_HASHES}\n    if strict_hashes: need(files==EXPECTED_HASHES,\'Input bytes differ from supplied v2-en.zip; do not silently evaluate another test.\')\n    manifest=strict_json((p/\'manifest.json\').read_text())\n    need(manifest[\'language\']==\'en\' and manifest[\'split_key\']==\'movie_id\',\'Invalid manifest language/split_key\')\n    need(manifest[\'system_prompt_sha256\']==SYSTEM_SHA,\'Unexpected system prompt hash\')\n    ids={}; prompts={}; records={}; report={\'file_sha256\':files,\'splits\':{}}\n    seen_messages={}; seen_prompt={}; facts_by_id={}\n    for split in EXPECTED_COUNTS:\n        rows=load_rows(p/f\'{split}.jsonl\'); need(len(rows)==EXPECTED_COUNTS[split],f\'{split}: bad example count\')\n        need(manifest[\'examples\'][split]==len(rows),f\'{split}: manifest examples mismatch\')\n        ids[split]=set(); labels=Counter(); kinds=Counter(); genre_counts=Counter(); char_lengths=[]\n        for i,r in enumerate(rows):\n            tag=f\'{split}:{i+1}\'\n            try:\n                need(set(r)=={\'messages\',\'provenance\'},\'Invalid top-level schema\')\n                msgs=r[\'messages\']; need(isinstance(msgs,list) and len(msgs)==3,\'Expected 3 messages\')\n                need([m[\'role\'] for m in msgs]==[\'system\',\'user\',\'assistant\'],\'Expected system/user/assistant\')\n                need(all(set(m)=={\'role\',\'content\'} and isinstance(m[\'content\'],str) and m[\'content\'] for m in msgs),\'Invalid message schema\')\n                need(sha_bytes(msgs[0][\'content\'].encode())==SYSTEM_SHA,\'System prompt mismatch\')\n                ctx,summary,history,q=parse_user(msgs[1][\'content\'])\n                need(isinstance(ctx,list) and len(ctx)==1 and summary==\'\' and history==[],\'Unexpected context/history schema\')\n                m=ctx[0]; need(isinstance(m,dict),\'Movie must be object\')\n                need(set(m)<= {\'movie_id\',\'title\',\'genres\',\'director\',\'cast\'} and {\'movie_id\',\'title\',\'genres\',\'cast\'}<=set(m),\'Invalid context movie fields\')\n                need(all(isinstance(m[k],str) and bool(m[k].strip()) for k in [\'movie_id\',\'title\']),\'Invalid ID/title\')\n                need(isinstance(m[\'genres\'],list) and all(isinstance(g,str) and g.strip() for g in m[\'genres\']),\'Invalid genres\')\n                need(isinstance(m[\'cast\'],list),\'Invalid cast\')\n                if \'director\' in m: need(isinstance(m[\'director\'],str) and bool(m[\'director\'].strip()),\'Invalid director\')\n                pr=r[\'provenance\']; need(isinstance(pr,dict),\'Invalid provenance\')\n                need(str(pr[\'source_movie_id\'])==m[\'movie_id\'],\'Provenance/context movie_id mismatch\')\n                need(pr[\'source_sha256\']==manifest[\'source_sha256\'],\'Catalog hash mismatch\')\n                need(pr[\'origin\']==manifest[\'origin\'] and pr[\'generator\']==manifest[\'version\'],\'Provenance generator/origin mismatch\')\n                key=m[\'movie_id\']; need(key not in facts_by_id or facts_by_id[key]==m,\'Conflicting context facts for same ID\')\n                facts_by_id[key]=m; ids[split].add(key)\n                ref=strict_json(msgs[-1][\'content\']); need(ref==canonical_response(r),\'Gold JSON does not match context-derived facts\')\n                _,kind,_=describe(r); labels[ref[\'status\']]+=1; kinds[kind]+=1; genre_counts.update(m[\'genres\'])\n                h=digest(msgs); ph=digest(msgs[:-1])\n                need(h not in seen_messages,f\'Duplicate conversation: {seen_messages.get(h)}\')\n                need(ph not in seen_prompt,f\'Duplicate/conflicting prompt: {seen_prompt.get(ph)}\')\n                seen_messages[h]=tag; seen_prompt[ph]=tag; char_lengths.append(sum(len(x[\'content\']) for x in msgs))\n            except Exception as e: raise ValueError(f\'{tag}: {e}\') from e\n        need(len(ids[split])==EXPECTED_MOVIES[split]==manifest[\'movies\'][split],f\'{split}: movie count mismatch\')\n        records[split]=rows\n        report[\'splits\'][split]={\'examples\':len(rows),\'movies\':len(ids[split]),\'status\':dict(labels),\n            \'question_type\':dict(kinds),\'genres\':dict(genre_counts),\'max_conversation_chars\':max(char_lengths)}\n    for a,b in [(\'train\',\'validation\'),(\'train\',\'test\'),(\'validation\',\'test\')]:\n        overlap=ids[a]&ids[b]; need(not overlap,f\'Movie leakage {a}/{b}: {sorted(overlap)}\')\n    report.update(duplicate_conversations=0,duplicate_prompts=0,cross_split_movie_ids=0,\n        note=\'All original rows are answered, director/genres only; no augmentation.\')\n    return records,report\n', encoding='utf-8')
print('Created common.py')


Created common.py


### encoding.py — Mask assistant và padding


In [21]:
(SCRIPTS/'encoding.py').write_text("from common import need\n\ndef encode_row(tok,row,max_length):\n    prompt=tok.apply_chat_template(row['messages'][:-1],tokenize=True,add_generation_prompt=True)\n    full=tok.apply_chat_template(row['messages'],tokenize=True,add_generation_prompt=False)\n    need(full[:len(prompt)]==prompt,'Chat template token prefix mismatch; refusing unsafe masking')\n    need(len(full)<=max_length,f'{len(full)} tokens > max_length={max_length}; no silent truncation allowed')\n    need(len(full)>len(prompt),'No assistant tokens')\n    # Supervise assistant JSON and its EOS only. Ignore template trailing newlines.\n    eos=next((i for i in range(len(prompt),len(full)) if full[i]==tok.eos_token_id),None)\n    need(eos is not None,'Assistant EOS missing')\n    labels=[-100]*len(prompt)+full[len(prompt):eos+1]+[-100]*(len(full)-eos-1)\n    need(len(labels)==len(full),'Label alignment failure')\n    return {'input_ids':full,'attention_mask':[1]*len(full),'labels':labels}\n\nclass AssistantCollator:\n    def __init__(self,tok): self.tok=tok\n    def __call__(self,features):\n        import torch\n        n=((max(len(f['input_ids']) for f in features)+7)//8)*8\n        return {\n          'input_ids':torch.tensor([f['input_ids']+[self.tok.pad_token_id]*(n-len(f['input_ids'])) for f in features],dtype=torch.long),\n          'attention_mask':torch.tensor([f['attention_mask']+[0]*(n-len(f['input_ids'])) for f in features],dtype=torch.long),\n          'labels':torch.tensor([f['labels']+[-100]*(n-len(f['input_ids'])) for f in features],dtype=torch.long)}\n", encoding='utf-8')
print('Created encoding.py')


Created encoding.py


### prepare.py — GPU/dependency smoke test, tải model và thống kê token


In [22]:
(SCRIPTS/'prepare.py').write_text("import os, sys, json, platform, subprocess, importlib.metadata as md\nfrom pathlib import Path\nfrom common import *\nimport torch, numpy as np\nimport bitsandbytes as bnb\nfrom transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments\nimport peft, accelerate, datasets\nfrom huggingface_hub import HfApi, snapshot_download\n\ncfg_path=Path(sys.argv[1]); cfg=json.loads(cfg_path.read_text()); root=Path(cfg['output'])\nroot.mkdir(parents=True,exist_ok=True)\nneed(torch.cuda.is_available(),'CUDA unavailable. Select Kaggle GPU T4 x2 and enable Internet.')\nneed(torch.cuda.device_count()>=1,'No visible GPU')\nversions={x:md.version(x) for x in ['torch','transformers','peft','accelerate','datasets','bitsandbytes','tokenizers','huggingface-hub','safetensors','numpy','pandas','pyarrow']}\nprint('Python',sys.version,'\\nVersions',json.dumps(versions,indent=2),flush=True)\nneed(versions['bitsandbytes']=='0.48.2','bitsandbytes version mismatch')\ngpus=[]\nfor i in range(torch.cuda.device_count()):\n    prop=torch.cuda.get_device_properties(i)\n    gpus.append({'index':i,'name':prop.name,'VRAM_GiB':round(prop.total_memory/2**30,2),'capability':list(torch.cuda.get_device_capability(i))})\n    # Real 4-bit forward/backward detects CUDA wheel/kernel problems before downloading 6 GB.\n    with torch.cuda.device(i):\n        layer=bnb.nn.Linear4bit(64,64,bias=False,compute_dtype=torch.float16,quant_type='nf4',compress_statistics=True).to(f'cuda:{i}')\n        x=torch.randn(2,64,device=f'cuda:{i}',dtype=torch.float16,requires_grad=True)\n        layer(x).float().sum().backward(); torch.cuda.synchronize(i)\n        del layer,x; torch.cuda.empty_cache()\nprint('GPUs',gpus,flush=True)\natomic_json(root/'environment.json',{'versions':versions,'python':sys.version,'platform':platform.platform(),\n    'cuda_runtime':torch.version.cuda,'gpus':gpus,\n    'nvidia_smi':subprocess.check_output(['nvidia-smi'],text=True)})\n(root/'pip-freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))\nrows,report=audit(cfg['data_dir'])\n# Pin the HF commit on the first run; reuse exactly that revision after a restore.\nlock_path=root/'model_lock.json'\nif lock_path.exists():\n    lock=json.loads(lock_path.read_text()); need(lock['model_id']==cfg['model_id'],'Different base model in restored run')\nelse:\n    revision=HfApi().model_info(cfg['model_id'],revision=cfg['requested_revision']).sha\n    lock={'model_id':cfg['model_id'],'revision':revision}; atomic_json(lock_path,lock)\nif cfg['requested_revision']!='main': need(cfg['requested_revision']==lock['revision'],'Requested revision differs from restored revision')\nmodel_dir=snapshot_download(cfg['model_id'],revision=lock['revision'],allow_patterns=[\n    '*.json','*.safetensors','*.model','merges.txt','vocab.json','tokenizer*','LICENSE*','README.md'])\ncfg['revision']=lock['revision']; cfg['model_path']=model_dir\ncfg['versions']=versions\ncfg['gpu_count']=torch.cuda.device_count()\ntok=AutoTokenizer.from_pretrained(model_dir,local_files_only=True)\nneed(tok.pad_token_id is not None and tok.eos_token_id is not None,'Missing pad/EOS token')\n(root/'tokenizer').mkdir(exist_ok=True); tok.save_pretrained(root/'tokenizer')\natomic_json(root/'tokenizer_fingerprint.json',{'chat_template_sha256':sha_bytes(tok.chat_template.encode()),'pad_token_id':tok.pad_token_id,'eos_token_id':tok.eos_token_id})\nfrom encoding import encode_row\nprepared=root/'prepared'; prepared.mkdir(exist_ok=True)\nfor split,examples in rows.items():\n    lengths=[]; targets=[]; prompts=[]; encoded=[]\n    for i,r in enumerate(examples):\n        try:\n            item=encode_row(tok,r,cfg['max_length']); encoded.append(item)\n            lengths.append(len(item['input_ids'])); targets.append(sum(x!=-100 for x in item['labels']))\n            prompts.append(len(item['input_ids'])-targets[-1])\n        except Exception as e: raise ValueError(f'{split}:{i+1}: tokenization error: {e}') from e\n    def stats(a): return {k:float(v) for k,v in zip(['min','p50','p90','p95','p99','max'],np.percentile(a,[0,50,90,95,99,100]))}\n    report['splits'][split]['tokens']={'conversation':stats(lengths),'supervised_assistant':stats(targets),'masked':stats(prompts)}\n    # Test token lengths are diagnostics only; never write test into training inputs.\n    if split!='test':\n        with (prepared/f'{split}.jsonl').open('w') as f:\n            for item in encoded: f.write(json.dumps(item)+'\\n')\n    if split=='train':\n        first=encoded[0]; n=next(i for i,x in enumerate(first['labels']) if x!=-100)\n        print('First supervised target:',tok.decode([x for x in first['labels'] if x!=-100]),flush=True)\n        need(all(x==-100 for x in first['labels'][:n]),'Prompt mask failure')\n        atomic_json(root/'mask_example.json',{'prompt_tokens':n,'target_tokens':sum(x!=-100 for x in first['labels']),\n            'decoded_target':tok.decode([x for x in first['labels'] if x!=-100])})\n# Generation budget fixed before baseline; do not tune it from test outcomes.\nmax_prompt=max(len(tok.apply_chat_template(r['messages'][:-1],tokenize=True,add_generation_prompt=True)) for r in rows['test'])\nneed(max_prompt+cfg['generation']['max_new_tokens']<=32768,'Prompt + generation exceed model context window')\nreport['test_generation_max_prompt_tokens']=max_prompt\natomic_json(root/'data_audit.json',report)\natomic_json(root/'data_manifest.original.json',json.loads((Path(cfg['data_dir'])/'manifest.json').read_text()))\natomic_json(root/'input_fingerprints.json',report['file_sha256'])\natomic_json(cfg_path,cfg)\nprint(json.dumps(report,indent=2),flush=True)\nprint('READY. Test labels will not select hyperparameters/checkpoints.',flush=True)\n", encoding='utf-8')
print('Created prepare.py')


Created prepare.py


### train.py — Train QLoRA một T4 và chọn best checkpoint


In [23]:
(SCRIPTS/'train.py').write_text("import json, sys, os, math, time\nfrom pathlib import Path\nfrom common import *\nfrom encoding import AssistantCollator\nimport torch\nfrom datasets import Dataset\nfrom transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig,Trainer,TrainingArguments,set_seed,TrainerCallback\nfrom transformers.trainer_utils import get_last_checkpoint\nfrom peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training\n\ncfg=json.loads(Path(sys.argv[1]).read_text()); root=Path(cfg['output'])\nneed(torch.cuda.device_count()==1,'Training must see ONE GPU; launch with CUDA_VISIBLE_DEVICES=0')\n# Guard against stale baselines/modified inputs before optimizing.\nfor name,expected in json.loads((root/'input_fingerprints.json').read_text()).items():\n    need(sha_file(Path(cfg['data_dir'])/name)==expected,f'Input changed after audit: {name}')\ntrain_contract={'revision':cfg['revision'],'train':cfg['train'],'max_length':cfg['max_length'],\n    'train_sha':sha_file(root/'prepared/train.jsonl'),'val_sha':sha_file(root/'prepared/validation.jsonl'),\n    'versions':cfg['versions'],'seed':cfg['seed'],\n    'scripts':{n:sha_file(Path(__file__).parent/n) for n in ['common.py','encoding.py','train.py']}}\ncontract_path=root/'training_contract.json'\nif contract_path.exists(): need(json.loads(contract_path.read_text())==train_contract,'Training contract changed; use a NEW RUN_NAME.')\nelse: atomic_json(contract_path,train_contract)\ncompleted=root/'train_complete.json'\nif completed.exists():\n    result=json.loads(completed.read_text())\n    need(result['contract_hash']==digest(train_contract),'Completion contract mismatch')\n    for n,h in result['adapter_files'].items(): need(sha_file(root/'adapter'/n)==h,f'Adapter changed: {n}')\n    print('Verified completed training; reusing saved best adapter.',flush=True); sys.exit(0)\nset_seed(cfg['seed'])\ntok=AutoTokenizer.from_pretrained(root/'tokenizer',local_files_only=True); tok.padding_side='right'\nquant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)\nmodel=AutoModelForCausalLM.from_pretrained(cfg['model_path'],quantization_config=quant,\n    torch_dtype=torch.float16,device_map={'':0},attn_implementation='sdpa',local_files_only=True)\nmodel.config.use_cache=False\nmodel=prepare_model_for_kbit_training(model,use_gradient_checkpointing=True,\n    gradient_checkpointing_kwargs={'use_reentrant':False})\nt=cfg['train']\nmodel=get_peft_model(model,LoraConfig(r=t['r'],lora_alpha=t['alpha'],lora_dropout=t['dropout'],\n    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],\n    bias='none',task_type='CAUSAL_LM'))\nmodel.print_trainable_parameters()\nneed(all(not p.requires_grad for n,p in model.named_parameters() if 'lora_' not in n),'Unexpected trainable base weights')\ntrain=Dataset.from_list(load_rows(root/'prepared/train.jsonl')); val=Dataset.from_list(load_rows(root/'prepared/validation.jsonl'))\nargs=TrainingArguments(output_dir=str(root/'checkpoints'),num_train_epochs=t['epochs'],\n    per_device_train_batch_size=1,per_device_eval_batch_size=1,gradient_accumulation_steps=t['grad_accum'],\n    learning_rate=t['learning_rate'],lr_scheduler_type='cosine',warmup_ratio=0.05,weight_decay=0.01,\n    max_grad_norm=1.0,fp16=True,bf16=False,tf32=False,optim='paged_adamw_8bit',\n    gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False},\n    eval_strategy='steps',eval_steps=t['eval_steps'],save_strategy='steps',save_steps=t['eval_steps'],\n    save_total_limit=3,load_best_model_at_end=True,metric_for_best_model='eval_loss',greater_is_better=False,\n    prediction_loss_only=True,logging_steps=10,logging_first_step=True,report_to='none',\n    seed=cfg['seed'],data_seed=cfg['seed'],dataloader_num_workers=0,remove_unused_columns=False,\n    save_safetensors=True,save_only_model=False)\nclass FiniteLoss(TrainerCallback):\n    def on_log(self,args,state,control,logs=None,**kwargs):\n        for k in ['loss','eval_loss','grad_norm']:\n            if logs and k in logs: need(math.isfinite(logs[k]),f'Non-finite {k}: {logs[k]}')\ntrainer=Trainer(model=model,args=args,train_dataset=train,eval_dataset=val,\n    data_collator=AssistantCollator(tok),processing_class=tok,callbacks=[FiniteLoss()])\nckpt=get_last_checkpoint(str(root/'checkpoints')) if (root/'checkpoints').exists() else None\nif ckpt:\n    need((Path(ckpt)/'trainer_state.json').exists(),'Incomplete checkpoint')\n    print('Resume trusted LOCAL checkpoint:',ckpt,flush=True)\nstarted=time.time(); result=trainer.train(resume_from_checkpoint=ckpt)\nneed(trainer.state.best_model_checkpoint is not None,'No validation checkpoint selected')\nneed(math.isfinite(trainer.state.best_metric),'Non-finite best validation loss')\n# Trainer has already loaded the best adapter, selected exclusively by validation loss.\nad=root/'adapter'; trainer.save_model(str(ad)); tok.save_pretrained(ad)\ntrainer.save_state(); trainer.save_metrics('train',result.metrics)\natomic_json(root/'trainer_history.json',trainer.state.log_history)\natomic_json(root/'training_arguments.json',args.to_dict())\nfiles={p.name:sha_file(p) for p in ad.iterdir() if p.is_file()}\natomic_json(completed,{'contract_hash':digest(train_contract),'best_checkpoint':trainer.state.best_model_checkpoint,\n    'best_validation_loss':trainer.state.best_metric,'seconds':time.time()-started,'adapter_files':files,\n    'effective_batch':t['grad_accum'],'test_used_for_selection':False})\nprint('Saved BEST adapter:',ad,flush=True)\n", encoding='utf-8')
print('Created train.py')


Created train.py


### evaluate.py — Worker đánh giá có resume theo mẫu


In [24]:
(SCRIPTS/'evaluate.py').write_text('"""One inference replica per visible GPU. No gold answers enter generate()."""\nimport json,sys,time,os,traceback\nfrom pathlib import Path\nfrom common import *\nimport torch\nfrom transformers import AutoModelForCausalLM,AutoTokenizer,BitsAndBytesConfig,set_seed,GenerationConfig\nfrom peft import PeftModel\nfrom tqdm import tqdm\n\ncfg=json.loads(Path(sys.argv[1]).read_text()); job=json.loads(Path(sys.argv[2]).read_text())\nrank=int(sys.argv[3]); workers=int(job[\'workers\']); root=Path(cfg[\'output\']); cache=Path(job[\'cache\'])\nneed(torch.cuda.device_count()==1,\'Each evaluation worker must see exactly one GPU\')\nrows=load_rows(Path(cfg[\'data_dir\'])/\'test.jsonl\')\nneed(sha_file(Path(cfg[\'data_dir\'])/\'test.jsonl\')==job[\'test_sha256\'],\'Test changed since evaluation job creation\')\nindices=job[\'indices\']; assigned=[i for i in indices if i%workers==rank]\ndef valid_saved(i):\n    path=cache/\'samples\'/f\'{i:04d}.json\'\n    if not path.exists(): return False\n    item=strict_json(path.read_text())\n    need(item[\'job_signature\']==job[\'signature\'] and item[\'sample_id\']==digest(rows[i][\'messages\'][:-1]),f\'Stale prediction at {path}\')\n    need(isinstance(item[\'raw_response\'],str),\'Corrupt prediction record\')\n    return True\npending=[i for i in assigned if not valid_saved(i)]\nprint(f"{job[\'mode\']} GPU worker {rank}: {len(assigned)-len(pending)} resumed, {len(pending)} pending",flush=True)\nif not pending: sys.exit(0)\nset_seed(cfg[\'seed\'])\ntok=AutoTokenizer.from_pretrained(root/\'tokenizer\',local_files_only=True)\nquant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type=\'nf4\',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)\nmodel=AutoModelForCausalLM.from_pretrained(cfg[\'model_path\'],quantization_config=quant,\n    torch_dtype=torch.float16,device_map={\'\':0},attn_implementation=\'sdpa\',local_files_only=True)\nif job[\'mode\']==\'adapter\': model=PeftModel.from_pretrained(model,root/\'adapter\',is_trainable=False)\nmodel.eval(); model.config.use_cache=True\n# Override inherited model generation defaults. Identical explicit config for both models.\ngen=GenerationConfig(do_sample=False,num_beams=1,max_new_tokens=cfg[\'generation\'][\'max_new_tokens\'],\n    repetition_penalty=1.0,use_cache=True,pad_token_id=tok.pad_token_id,eos_token_id=tok.eos_token_id,\n    bos_token_id=tok.bos_token_id)\nwith torch.inference_mode():\n    for i in tqdm(assigned,desc=f"{job[\'mode\']} GPU{rank}",position=0,dynamic_ncols=True):\n        if valid_saved(i): continue\n        row=rows[i]; sample_id=digest(row[\'messages\'][:-1])\n        # Order/resume/worker count do not change a sample\'s seed.\n        seed=(cfg[\'seed\']+int(sample_id[:8],16))%(2**31); set_seed(seed)\n        prompt=tok.apply_chat_template(row[\'messages\'][:-1],tokenize=False,add_generation_prompt=True)\n        inputs=tok(prompt,return_tensors=\'pt\',add_special_tokens=False).to(\'cuda:0\')\n        start=time.time()\n        try:\n            out=model.generate(**inputs,generation_config=gen)\n            generated=out[0,inputs[\'input_ids\'].shape[1]:]\n            raw=tok.decode(generated,skip_special_tokens=True)\n            record={\'index\':i,\'sample_id\':sample_id,\'mode\':job[\'mode\'],\'job_signature\':job[\'signature\'],\n              \'seed\':seed,\'raw_response\':raw,\'prompt_sha256\':sha_bytes(prompt.encode()),\n              \'prompt_tokens\':inputs[\'input_ids\'].shape[1],\'generated_tokens\':len(generated),\n              \'hit_token_limit\':bool(len(generated)>=gen.max_new_tokens and generated[-1].item()!=tok.eos_token_id),\n              \'seconds\':time.time()-start}\n            atomic_json(cache/\'samples\'/f\'{i:04d}.json\',record)\n            # A second, human-readable append log; authoritative resume uses atomic per-sample files.\n            with (cache/f\'worker_{rank}.jsonl\').open(\'a\',encoding=\'utf-8\') as f:\n                f.write(json.dumps(record,ensure_ascii=False)+\'\\n\'); f.flush(); os.fsync(f.fileno())\n        except Exception:\n            print(f\'FATAL mode={job["mode"]} rank={rank} index={i} movie_id={row["provenance"]["source_movie_id"]}\',file=sys.stderr,flush=True)\n            traceback.print_exc(); raise\n', encoding='utf-8')
print('Created evaluate.py')


Created evaluate.py


### report.py — Báo cáo paired và phân tích lỗi


In [25]:
(SCRIPTS/'report.py').write_text('import sys,json,csv,random,math\nfrom pathlib import Path\nfrom collections import Counter,defaultdict\nfrom common import *\n\ncfg=json.loads(Path(sys.argv[1]).read_text()); root=Path(cfg[\'output\']); rows=load_rows(Path(cfg[\'data_dir\'])/\'test.jsonl\')\nbase_job=json.loads((root/\'base_job.json\').read_text()); adapter_job=json.loads((root/\'adapter_job.json\').read_text())\nneed(base_job[\'indices\']==adapter_job[\'indices\'],\'Different test subsets\')\nneed(base_job[\'contract\']==adapter_job[\'contract\'],\'Base/adapter evaluation contracts differ\')\nindices=base_job[\'indices\']; full=len(indices)==198 and indices==list(range(198))\nreports=root/\'reports\'; reports.mkdir(exist_ok=True)\nall_records={}; stats={}\nfor mode,job in [(\'base\',base_job),(\'adapter\',adapter_job)]:\n    recs=[]\n    for i in indices:\n        path=Path(job[\'cache\'])/\'samples\'/f\'{i:04d}.json\'; need(path.exists(),f\'Missing {mode} prediction {i}\')\n        pred=strict_json(path.read_text()); need(pred[\'job_signature\']==job[\'signature\'],\'Prediction contract mismatch\')\n        need(pred[\'sample_id\']==digest(rows[i][\'messages\'][:-1]),\'Sample hash mismatch\')\n        ctx,kind,q=describe(rows[i]); check=score(rows[i],pred[\'raw_response\'])\n        recs.append({**pred,\'question_type\':kind,\'question\':q,\'context\':ctx,\'reference\':canonical_response(rows[i]),**check})\n    all_records[mode]={r[\'index\']:r for r in recs}\n    # Always export one clean canonical JSONL, even if append-only worker logs were interrupted.\n    (reports/f\'{mode}_scored.jsonl\').write_text(\'\'.join(json.dumps(r,ensure_ascii=False)+\'\\n\' for r in recs),encoding=\'utf-8\')\n    stats[mode]={\'n\':len(recs),\'full_original_test\':full,\'metrics\':{k:{\'passed\':sum(r[k] for r in recs),\'rate\':sum(r[k] for r in recs)/len(recs)} for k in METRICS},\n        \'by_question_type\':{kind:{\'n\':sum(r[\'question_type\']==kind for r in recs),\n          \'all_checks\':sum(r[\'all_checks\'] for r in recs if r[\'question_type\']==kind)} for kind in [\'director\',\'genres\']},\n        \'failure_counts\':dict(Counter(x for r in recs for x in r[\'failure_reasons\'])),\n        \'token_limit_hits\':sum(r[\'hit_token_limit\'] for r in recs)}\n    errors=[r for r in recs if not r[\'all_checks\']]\n    (reports/f\'{mode}_errors.jsonl\').write_text(\'\'.join(json.dumps(r,ensure_ascii=False)+\'\\n\' for r in errors),encoding=\'utf-8\')\npaired=[]\nfor i in indices:\n    a=all_records[\'base\'][i]; b=all_records[\'adapter\'][i]\n    need(a[\'prompt_sha256\']==b[\'prompt_sha256\'] and a[\'seed\']==b[\'seed\'],\'Prompt/seed mismatch\')\n    transition=(\'both_pass\' if a[\'all_checks\'] and b[\'all_checks\'] else \'improved\' if b[\'all_checks\'] else \'regressed\' if a[\'all_checks\'] else \'both_fail\')\n    paired.append({\'index\':i,\'movie_id\':a[\'context\'][0][\'movie_id\'],\'question_type\':a[\'question_type\'],\n        \'question\':a[\'question\'],\'transition\':transition,\'base_all_checks\':a[\'all_checks\'],\'adapter_all_checks\':b[\'all_checks\'],\n        \'base_errors\':\'; \'.join(a[\'failure_reasons\']),\'adapter_errors\':\'; \'.join(b[\'failure_reasons\']),\n        \'base_response\':a[\'raw_response\'],\'adapter_response\':b[\'raw_response\'],\n        \'context\':json.dumps(a[\'context\'],ensure_ascii=False),\'reference\':json.dumps(a[\'reference\'],ensure_ascii=False)})\nwith (reports/\'paired_comparison.csv\').open(\'w\',newline=\'\',encoding=\'utf-8-sig\') as f:\n    w=csv.DictWriter(f,fieldnames=list(paired[0])); w.writeheader(); w.writerows(paired)\ntransitions=dict(Counter(p[\'transition\'] for p in paired))\n# Movie-level cluster bootstrap: questions for one film are not independent observations.\ngroups=defaultdict(list)\nfor p in paired: groups[p[\'movie_id\']].append(int(p[\'adapter_all_checks\'])-int(p[\'base_all_checks\']))\nrng=random.Random(cfg[\'seed\']); keys=sorted(groups); deltas=[]\nfor _ in range(2000):\n    sampled=[x for key in rng.choices(keys,k=len(keys)) for x in groups[key]]\n    deltas.append(sum(sampled)/len(sampled))\ndeltas.sort(); delta_ci=[deltas[int(.025*len(deltas))],deltas[int(.975*len(deltas))]]\nsummary={\'evaluation_scope\':\'FULL ORIGINAL TEST\' if full else \'QUICK SUBSET — NOT A FULL TEST RESULT\',\n    \'models\':stats,\'paired_transitions\':transitions,\n    \'all_checks_delta\':stats[\'adapter\'][\'metrics\'][\'all_checks\'][\'rate\']-stats[\'base\'][\'metrics\'][\'all_checks\'][\'rate\'],\n    \'movie_cluster_bootstrap_95pct_delta_interval\':delta_ci,\n    \'facts_metric\':\'Strict normalized context-derived answer AND reason equality; a conservative template proxy, not semantic entailment.\',\n    \'limitations\':[\'Only answered labels; status accuracy cannot measure refusal/no_match/insufficient_context.\',\n      \'Only one-film director/genres questions; no overview, multi-film recommendation, memory or prompt-injection benchmark.\',\n      \'Valid factual paraphrases can fail facts_correct; inspect paired CSV manually.\',\n      \'Both models evaluated with NF4 quantization, not full-precision base.\',\n      \'One training seed; bootstrap interval measures test-film sampling, not training-seed variance.\',\n      \'No conclusions about Vietnamese or real-user production reliability.\']}\natomic_json(reports/\'summary.json\',summary)\nlines=[\'# WhatToWatch — \'+summary[\'evaluation_scope\'],\'\',\n    \'| Metric | Base | Adapter | Delta (percentage points) |\',\'|---|---:|---:|---:|\']\nfor k in METRICS:\n    a=stats[\'base\'][\'metrics\'][k];b=stats[\'adapter\'][\'metrics\'][k]\n    lines.append(f"| {k} | {a[\'passed\']}/{len(indices)} ({a[\'rate\']:.1%}) | {b[\'passed\']}/{len(indices)} ({b[\'rate\']:.1%}) | {100*(b[\'rate\']-a[\'rate\']):+.2f} |")\nlines+=[\'\',\'Transitions: \'+json.dumps(transitions),\n    \'\',f\'Movie-cluster bootstrap 95% interval for all_checks delta: [{delta_ci[0]:.3f}, {delta_ci[1]:.3f}].\',\n    \'\',\'## Interpretation\',\'\',summary[\'facts_metric\'],\'\']+[\'- \'+x for x in summary[\'limitations\']]\nfor mode in [\'base\',\'adapter\']:\n    lines+=[\'\',f\'## {mode}: failures by question type\',\'\',json.dumps(stats[mode][\'by_question_type\']),\n        \'\', \'Failure counts: \'+json.dumps(stats[mode][\'failure_counts\']),\n        \'\',f\'Token-limit hits: {stats[mode]["token_limit_hits"]}.\']\n    for r in [r for r in all_records[mode].values() if not r[\'all_checks\']][:5]:\n        lines+=[\'\',f\'### Sample {r["index"]}: {r["question"]}\',\'\',\n            \'Failed: \'+\', \'.join(r[\'failure_reasons\']),\'\',\'```json\',r[\'raw_response\'],\'```\',\'\',\n            \'Reference:\',\'```json\',json.dumps(r[\'reference\'],ensure_ascii=False),\'```\']\n(reports/\'report.md\').write_text(\'\\n\'.join(lines),encoding=\'utf-8\')\nprint(\'\\n\'.join(lines[:20]),flush=True)\nprint(\'Detailed errors, paired CSV, and limitations:\',reports,flush=True)\n', encoding='utf-8')
print('Created report.py')


Created report.py


## 4. Chạy kiểm tra đầu vào, GPU và tokenization
Đọc cấu trúc tất cả split để audit/hash/độ dài; **test không được đưa vào Trainer**. Các mục kiểm tra: role/schema, hash system prompt, gold nhất quán với context, số lượng theo manifest, câu hỏi được hỗ trợ, ID trong provenance, dữ kiện nhất quán, duplicate conversation/prompt, movie leakage và độ dài token. Không truncation âm thầm.

Kiểm tra CUDA gồm forward/backward NF4 thực trên mỗi GPU. Nếu lỗi, xem log ở `logs/prepare.log`; không đổi ngẫu nhiên phiên bản bitsandbytes. Tokenizer được khóa cùng commit model. Số lượng nhãn/thể loại và phân vị token xuất vào `data_audit.json`.


In [27]:
import os
import sys
import subprocess

# Các subprocess tiếp theo cũng nhận cấu hình này.
os.environ.update({
    "USE_TORCH": "1",
    "USE_TF": "0",
    "USE_FLAX": "0",
})

diagnostic = r"""
import sys
import importlib.metadata as metadata

print("Python:", sys.version, flush=True)

for package in [
    "torch", "transformers", "peft", "accelerate",
    "bitsandbytes", "numpy", "torchvision", "librosa"
]:
    try:
        print(f"{package}: {metadata.version(package)}", flush=True)
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed", flush=True)

print("\n[1] Import torch", flush=True)
import torch
print("CUDA runtime:", torch.version.cuda, flush=True)
print("CUDA available:", torch.cuda.is_available(), flush=True)

print("\n[2] Import trực tiếp modeling_utils", flush=True)
from transformers.modeling_utils import PreTrainedModel

print("\n[3] Import Trainer / PEFT", flush=True)
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training

print("\nIMPORT CHECK PASSED", flush=True)
"""

result = subprocess.run(
    [sys.executable, "-u", "-c", diagnostic],
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(result.stdout)

if result.returncode != 0:
    print("IMPORT CHECK FAILED — xem traceback gốc phía trên.")
else:
    print("Import thành công. Chạy lại cell stream_process.")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
torch: 2.6.0+cu124
transformers: 4.53.3
peft: 0.17.1
accelerate: 1.10.1
bitsandbytes: 0.48.2
numpy: 1.26.4
torchvision: not installed
librosa: not installed

[1] Import torch
CUDA runtime: 12.4
CUDA available: True

[2] Import trực tiếp modeling_utils

[3] Import Trainer / PEFT

IMPORT CHECK PASSED

Import thành công. Chạy lại cell stream_process.


In [28]:
# Notebook kernel remains CPU/stdlib-only. Fresh child processes own GPU memory.
LOGS=ROOT/'logs'; LOGS.mkdir(exist_ok=True)

def stream_process(command,log_name,env=None):
    logfile=LOGS/log_name
    child_env=os.environ.copy(); child_env.update({
        'PYTHONUNBUFFERED':'1','TOKENIZERS_PARALLELISM':'false',
        'WANDB_DISABLED':'true','HF_HUB_DISABLE_TELEMETRY':'1',
        'CUBLAS_WORKSPACE_CONFIG':':4096:8','PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'})
    if env: child_env.update(env)
    print('Launch:',command,'\nLog:',logfile,flush=True)
    with logfile.open('a',encoding='utf-8') as log:
        proc=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
            text=True,bufsize=1,env=child_env)
        try:
            for line in proc.stdout:
                print(line,end='',flush=True);log.write(line);log.flush()
            rc=proc.wait()
        except BaseException:
            proc.terminate()
            try: proc.wait(timeout=10)
            except subprocess.TimeoutExpired: proc.kill();proc.wait()
            raise
    if rc:
        tail='\n'.join(logfile.read_text(errors='replace').splitlines()[-70:])
        raise RuntimeError(f'Subprocess failed (exit {rc}): {command}\nLog: {logfile}\n{tail}')

stream_process([sys.executable,'-u',str(SCRIPTS/'prepare.py'),str(CONFIG)],'prepare.log')
cfg=json.loads(CONFIG.read_text())
audit_report=json.loads((ROOT/'data_audit.json').read_text())
for split,item in audit_report['splits'].items():
    print(split,'examples=',item['examples'],'movies=',item['movies'],
          'status=',item['status'],'questions=',item['question_type'],'tokens=',item['tokens'])
print('Training sees train/validation only. Full original test labels are not checkpoint criteria.')


Launch: ['/usr/bin/python3', '-u', '/kaggle/working/w2w_english_v1/scripts/prepare.py', '/kaggle/working/w2w_english_v1/config.json'] 
Log: /kaggle/working/w2w_english_v1/logs/prepare.log
Python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0] 
Versions {
  "torch": "2.6.0+cu124",
  "transformers": "4.53.3",
  "peft": "0.17.1",
  "accelerate": "1.10.1",
  "datasets": "3.6.0",
  "bitsandbytes": "0.48.2",
  "tokenizers": "0.21.4",
  "huggingface-hub": "0.34.4",
  "safetensors": "0.6.2",
  "numpy": "1.26.4",
  "pandas": "2.2.3",
  "pyarrow": "20.0.0"
}
GPUs [{'index': 0, 'name': 'Tesla T4', 'VRAM_GiB': 14.56, 'capability': [7, 5]}, {'index': 1, 'name': 'Tesla T4', 'VRAM_GiB': 14.56, 'capability': [7, 5]}]

Fetching 11 files: 100%|██████████| 11/11 [00:27<00:00,  2.54s/it]
First supervised target: {"status": "answered", "answer": "The director is Andrew Traucki.", "movies": [{"movie_id": "26399", "title": "The Reef", "reason": "Director: Andrew Traucki"}]}<|im_end|>
{
  "file_sha256": {


## 5. Bộ điều phối đánh giá có cache và tiến trình trực tiếp
Hai worker độc lập, mỗi worker chỉ nhìn thấy một GPU; không dùng `device_map="auto"` hoặc DataParallel. Mỗi mẫu ghi một JSON **atomic + fsync**, kèm log JSONL. Cache chỉ tái dùng nếu commit model, tokenizer, input, cấu hình sinh, seed, dependency versions, script và adapter fingerprint khớp.

Đổi từ 2 GPU sang 1 GPU vẫn có thể tiếp tục cache. `base_predictions.jsonl`/`adapter_predictions.jsonl` được hợp nhất từ các record atomic sau mỗi lần chạy, kể cả khi worker báo lỗi. Log lỗi giữ traceback và return code. Nếu ngắt notebook bằng Interrupt, subprocess được dừng; mẫu chưa hoàn tất sẽ chạy lại.


In [29]:
def stable_hash(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
def file_hash(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()

def evaluate(mode):
    assert mode in ['base','adapter']
    n=198; limit=cfg['eval_limit']
    if limit is not None:
        if not isinstance(limit,int) or isinstance(limit,bool) or not 1<=limit<=n:
            raise ValueError('EVAL_LIMIT must be None or an integer in [1,198]')
        import random
        indices=sorted(random.Random(cfg['seed']).sample(range(n),limit))
    else: indices=list(range(n))
    versions=json.loads((ROOT/'environment.json').read_text())['versions']
    contract={
        'model_id':cfg['model_id'],'revision':cfg['revision'],
        'test_sha256':file_hash(DATA_DIR/'test.jsonl'),'indices':indices,
        'generation':cfg['generation'],'seed':cfg['seed'],
        'quantization':{'bits':4,'type':'nf4','double_quant':True,'compute_dtype':'float16'},
        'attention':'sdpa','batch_size':1,'versions':versions,
        'tokenizer':json.loads((ROOT/'tokenizer_fingerprint.json').read_text()),
        'scripts':{n:file_hash(SCRIPTS/n) for n in ['common.py','evaluate.py']},
    }
    adapter_hash=None
    if mode=='adapter':
        if not (ROOT/'train_complete.json').exists(): raise ValueError('Training has not completed')
        completion=json.loads((ROOT/'train_complete.json').read_text())
        for name,h in completion['adapter_files'].items():
            if file_hash(ROOT/'adapter'/name)!=h: raise ValueError('Adapter fingerprint changed: '+name)
        adapter_hash=stable_hash(completion['adapter_files'])
    signature=stable_hash({'contract':contract,'mode':mode,'adapter_hash':adapter_hash})
    cache=ROOT/'eval_cache'/mode/signature
    (cache/'samples').mkdir(parents=True,exist_ok=True)
    workers=min(2,cfg['gpu_count'])
    job={'contract':contract,'mode':mode,'adapter_hash':adapter_hash,'signature':signature,
         'cache':str(cache),'indices':indices,'workers':workers,'test_sha256':contract['test_sha256']}
    jobpath=ROOT/f'{mode}_job.json'; jobpath.write_text(json.dumps(job,indent=2))
    errors=[]; processes=[]; threads=[]
    def relay(proc,rank):
        with (LOGS/f'{mode}_gpu{rank}.log').open('a',encoding='utf-8') as f:
            for line in proc.stdout:
                print(f'[{mode}/GPU{rank}] '+line,end='',flush=True);f.write(line);f.flush()
    def export_predictions():
        records=[]
        for i in indices:
            p=cache/'samples'/f'{i:04d}.json'
            if p.exists():
                r=json.loads(p.read_text())
                if r['job_signature']!=signature: raise ValueError('Invalid cached signature')
                records.append(r)
        tmp=ROOT/f'{mode}_predictions.jsonl.tmp'
        tmp.write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in records),encoding='utf-8')
        os.replace(tmp,ROOT/f'{mode}_predictions.jsonl')
        return len(records)
    try:
        for rank in range(workers):
            env=os.environ.copy(); env.update({
              'CUDA_VISIBLE_DEVICES':str(rank),'PYTHONUNBUFFERED':'1',
              'TOKENIZERS_PARALLELISM':'false','HF_HUB_DISABLE_TELEMETRY':'1',
              'CUBLAS_WORKSPACE_CONFIG':':4096:8'})
            command=[sys.executable,'-u',str(SCRIPTS/'evaluate.py'),str(CONFIG),str(jobpath),str(rank)]
            p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
            processes.append(p)
            thread=threading.Thread(target=relay,args=(p,rank),daemon=True);thread.start();threads.append(thread)
        while any(p.poll() is None for p in processes):
            failed=[(i,p.returncode) for i,p in enumerate(processes) if p.poll() is not None and p.returncode!=0]
            if failed:
                errors.extend(failed)
                for p in processes:
                    if p.poll() is None: p.terminate()
                break
            time.sleep(0.2)
        for i,p in enumerate(processes):
            rc=p.wait()
            if rc!=0 and (i,rc) not in errors: errors.append((i,rc))
    finally:
        for p in processes:
            if p.poll() is None:
                p.terminate()
                try: p.wait(timeout=10)
                except subprocess.TimeoutExpired: p.kill();p.wait()
        for t in threads: t.join(timeout=10)
        count=export_predictions()
        print(f'{mode}: {count}/{len(indices)} atomic records exported',flush=True)
    if errors:
        tails=[]
        for rank,rc in errors:
            p=LOGS/f'{mode}_gpu{rank}.log'
            tails.append(f'GPU {rank}, exit={rc}, log={p}\n'+'\n'.join(p.read_text(errors='replace').splitlines()[-50:]))
        raise RuntimeError('Evaluation subprocess failed:\n'+'\n'.join(tails))
    if count!=len(indices): raise RuntimeError(f'Incomplete evaluation: {count}/{len(indices)}')
    return job


## 6. Baseline trên test cố định — trước train
Chạy inference model gốc **NF4** trên test gốc. Để giữ test làm đánh giá cuối, cell này chỉ tạo dự đoán và thống kê tiến độ, **chưa mở điểm test**. Báo cáo định lượng sẽ chạy sau khi checkpoint tốt nhất đã được chọn bằng validation loss. Không xem lỗi baseline để điều chỉnh hyperparameter cho cùng test.


In [31]:
import os
import sys
import json
import subprocess

os.environ.update({
    "USE_TORCH": "1",
    "USE_TF": "0",
    "USE_FLAX": "0",
})

# Workflow text-only không cần timm.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "timm"],
    check=True,
)

# Kiểm tra lại và cập nhật thông tin môi trường.
stream_process(
    [sys.executable, "-u", str(SCRIPTS / "prepare.py"), str(CONFIG)],
    "prepare.log",
)
cfg = json.loads(CONFIG.read_text())

# Chạy lại baseline.
base_job = evaluate("base")
print("Baseline predictions saved. Continue to validation-selected training.")

Found existing installation: timm 1.0.29
Uninstalling timm-1.0.29:
  Successfully uninstalled timm-1.0.29
Launch: ['/usr/bin/python3', '-u', '/kaggle/working/w2w_english_v1/scripts/prepare.py', '/kaggle/working/w2w_english_v1/config.json'] 
Log: /kaggle/working/w2w_english_v1/logs/prepare.log
Python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0] 
Versions {
  "torch": "2.6.0+cu124",
  "transformers": "4.53.3",
  "peft": "0.17.1",
  "accelerate": "1.10.1",
  "datasets": "3.6.0",
  "bitsandbytes": "0.48.2",
  "tokenizers": "0.21.4",
  "huggingface-hub": "0.34.4",
  "safetensors": "0.6.2",
  "numpy": "1.26.4",
  "pandas": "2.2.3",
  "pyarrow": "20.0.0"
}
GPUs [{'index': 0, 'name': 'Tesla T4', 'VRAM_GiB': 14.56, 'capability': [7, 5]}, {'index': 1, 'name': 'Tesla T4', 'VRAM_GiB': 14.56, 'capability': [7, 5]}]

Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 129599.28it/s]
First supervised target: {"status": "answered", "answer": "The director is Andrew Traucki.", "movies": [{"mo

## 7. Fine-tune QLoRA một T4
| Thiết lập | Mặc định | Lý do |
|---|---|---|
| Quantization | NF4 4-bit + double quant, FP16 compute | Giảm VRAM; T4 dùng FP16, không dùng BF16 |
| LoRA | r=16, alpha=32, dropout=0.05 | Điểm khởi đầu vừa phải; giữ cấu hình tham khảo cũ, chưa khẳng định tối ưu |
| Target modules | Q/K/V/O + gate/up/down | Thích nghi attention và MLP; base frozen |
| Batch | 1 × accumulation 8 × 1 GPU = 8 | Phù hợp VRAM một T4, GPU1 dành đánh giá |
| Sequence | tối đa 1024, dynamic padding | Báo lỗi nếu quá dài, không cắt mất JSON/context |
| Learning rate | 1e-4, cosine, warmup 5% | Điểm khởi đầu cho adapter trên tập nhỏ |
| Epoch | 2 | Tránh mặc định train quá lâu trên dữ liệu template |
| Checkpoint | mỗi 50 optimizer steps, giữ 3 | Có state để resume; best theo validation loss |
| Loss | assistant JSON + EOS, pad/prompt = -100 | Không học dự đoán lại system/user |
| Memory | gradient checkpointing, paged AdamW 8-bit, SDPA | Giảm activation/optimizer memory |

Không packing, không augmentation, không dịch tiếng Việt. Với 1.613 mẫu, khoảng 202 optimizer steps/epoch và khoảng 404 steps tổng (Trainer quyết định chính xác theo batch cuối). Tốc độ phụ thuộc runtime; không đưa thời gian giả định thành cam kết.

Train được cô lập bằng `CUDA_VISIBLE_DEVICES=0`; không để Trainer tự thấy hai GPU rồi chọn DataParallel. Validation chỉ gồm 185 mẫu gốc. Test không được tải vào `train.py` ngoài việc kiểm tra checksum file.


In [32]:
# Baseline must be complete before training.
if len(list((Path(base_job['cache'])/'samples').glob('*.json')))!=len(base_job['indices']):
    raise ValueError('Complete baseline evaluation before training')
stream_process([sys.executable,'-u',str(SCRIPTS/'train.py'),str(CONFIG)],'train.log',
               env={'CUDA_VISIBLE_DEVICES':'0'})
print(json.dumps(json.loads((ROOT/'train_complete.json').read_text()),indent=2))


Launch: ['/usr/bin/python3', '-u', '/kaggle/working/w2w_english_v1/scripts/train.py', '/kaggle/working/w2w_english_v1/config.json'] 
Log: /kaggle/working/w2w_english_v1/logs/train.log

Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.92s/it]
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.

  0%|          | 1/404 [00:09<1:06:51,  9.95s/it]
                                                 
{'loss': 0.7061, 'grad_norm': 5.915341854095459, 'learning_rate': 0.0, 'epoch': 0.0}

  2%|▏         | 10/404 [01:31<59:31,  9.07s/it]
                                                
{'loss': 0.3738, 'grad_norm': 1.279698371887207, 'learning_rate': 4.2857142857142856e-05, 'epoch': 0.05}

  5%|▍       

## 8. Đánh giá best adapter trên đúng cùng test
Base và adapter cùng prompt/tokenizer/base commit, NF4, greedy decoding, max_new_tokens=256, batch=1, seed theo sample. Không sampling, không constrained JSON decoding, không sửa hay loại các câu trả lời sai JSON. Nếu bị token limit, lỗi vẫn tính trong mẫu số.


In [35]:
adapter_job=evaluate('adapter')


[adapter/GPU1] adapter GPU worker 1: 99 resumed, 0 pending
[adapter/GPU0] adapter GPU worker 0: 99 resumed, 0 pending
adapter: 198/198 atomic records exported


## 9. Chấm điểm, đối chiếu theo mẫu và phân tích lỗi
`json_valid`: JSON parser nghiêm ngặt, không Markdown, không duplicate key/NaN. `schema_valid`: đúng khóa/type và ràng buộc list theo status. `status_correct`: so gold. `movie_id_correct`: đúng ID kỳ vọng (string, không ép kiểu). `movie_id_in_context`: mọi ID có trong context, list không rỗng. `title_correct`: title đúng cùng movie ID.

**`facts_correct` là proxy chặt theo template**: toàn bộ answer và reason phải bằng chuỗi dựng từ context sau chuẩn hóa Unicode/case/whitespace/dấu cuối. Không chỉ kiểm tra tên đạo diễn xuất hiện đâu đó, vì cách đó có thể cho qua dữ kiện bịa kèm theo. **Proxy này có false negative với paraphrase đúng**, không phải thước đo semantic entailment tổng quát. `all_checks` là AND của mọi kiểm tra trên; không nên diễn giải thành độ tin cậy tổng thể của trợ lý.

Xuất summary JSON, report Markdown, scored/error JSONL và paired CSV chứa context/reference/hai câu trả lời. CSV dùng để kiểm tra thủ công các paraphrase bị đánh trượt và so sánh lỗi được sửa/lỗi mới. Bootstrap nhóm theo movie_id vì hai câu hỏi cùng phim không độc lập.


In [36]:
stream_process([sys.executable,'-u',str(SCRIPTS/'report.py'),str(CONFIG)],'report.log')
from IPython.display import display, Markdown
summary=json.loads((ROOT/'reports/summary.json').read_text())
display(Markdown((ROOT/'reports/report.md').read_text()))
print('CSV for manual review:',ROOT/'reports/paired_comparison.csv')


Launch: ['/usr/bin/python3', '-u', '/kaggle/working/w2w_english_v1/scripts/report.py', '/kaggle/working/w2w_english_v1/config.json'] 
Log: /kaggle/working/w2w_english_v1/logs/report.log
# WhatToWatch — FULL ORIGINAL TEST

| Metric | Base | Adapter | Delta (percentage points) |
|---|---:|---:|---:|
| json_valid | 198/198 (100.0%) | 198/198 (100.0%) | +0.00 |
| schema_valid | 131/198 (66.2%) | 198/198 (100.0%) | +33.84 |
| status_correct | 198/198 (100.0%) | 198/198 (100.0%) | +0.00 |
| movie_id_correct | 172/198 (86.9%) | 198/198 (100.0%) | +13.13 |
| movie_id_in_context | 172/198 (86.9%) | 198/198 (100.0%) | +13.13 |
| title_correct | 171/198 (86.4%) | 198/198 (100.0%) | +13.64 |
| facts_correct | 0/198 (0.0%) | 198/198 (100.0%) | +100.00 |
| all_checks | 0/198 (0.0%) | 198/198 (100.0%) | +100.00 |

Transitions: {"improved": 198}

Movie-cluster bootstrap 95% interval for all_checks delta: [1.000, 1.000].

## Interpretation

Strict normalized context-derived answer AND reason equality; 

# WhatToWatch — FULL ORIGINAL TEST

| Metric | Base | Adapter | Delta (percentage points) |
|---|---:|---:|---:|
| json_valid | 198/198 (100.0%) | 198/198 (100.0%) | +0.00 |
| schema_valid | 131/198 (66.2%) | 198/198 (100.0%) | +33.84 |
| status_correct | 198/198 (100.0%) | 198/198 (100.0%) | +0.00 |
| movie_id_correct | 172/198 (86.9%) | 198/198 (100.0%) | +13.13 |
| movie_id_in_context | 172/198 (86.9%) | 198/198 (100.0%) | +13.13 |
| title_correct | 171/198 (86.4%) | 198/198 (100.0%) | +13.64 |
| facts_correct | 0/198 (0.0%) | 198/198 (100.0%) | +100.00 |
| all_checks | 0/198 (0.0%) | 198/198 (100.0%) | +100.00 |

Transitions: {"improved": 198}

Movie-cluster bootstrap 95% interval for all_checks delta: [1.000, 1.000].

## Interpretation

Strict normalized context-derived answer AND reason equality; a conservative template proxy, not semantic entailment.

- Only answered labels; status accuracy cannot measure refusal/no_match/insufficient_context.
- Only one-film director/genres questions; no overview, multi-film recommendation, memory or prompt-injection benchmark.
- Valid factual paraphrases can fail facts_correct; inspect paired CSV manually.
- Both models evaluated with NF4 quantization, not full-precision base.
- One training seed; bootstrap interval measures test-film sampling, not training-seed variance.
- No conclusions about Vietnamese or real-user production reliability.

## base: failures by question type

{"director": {"n": 99, "all_checks": 0}, "genres": {"n": 99, "all_checks": 0}}

Failure counts: {"facts_correct": 198, "schema_valid": 67, "movie_id_correct": 26, "movie_id_in_context": 26, "title_correct": 27}

Token-limit hits: 0.

### Sample 0: Who directed Tattooed Life?

Failed: facts_correct

```json
{
  "status": "answered",
  "answer": "Seijun Suzuki",
  "movies": [
    {
      "movie_id": "4763",
      "title": "Tattooed Life",
      "reason": "The movie's director is Seijun Suzuki, as stated in the context."
    }
  ]
}
```

Reference:
```json
{"status": "answered", "answer": "The director is Seijun Suzuki.", "movies": [{"movie_id": "4763", "title": "Tattooed Life", "reason": "Director: Seijun Suzuki"}]}
```

### Sample 1: What genres does Tattooed Life belong to?

Failed: schema_valid, movie_id_correct, movie_id_in_context, title_correct, facts_correct

```json
{
  "status": "answered",
  "answer": [
    {
      "movie_id": "4763",
      "title": "Tattooed Life",
      "reason": "Crime, Drama, Thriller"
    }
  ]
}
```

Reference:
```json
{"status": "answered", "answer": "The movie belongs to these genres: Crime, Drama, Thriller.", "movies": [{"movie_id": "4763", "title": "Tattooed Life", "reason": "Genres: Crime, Drama, Thriller"}]}
```

### Sample 2: Who directed Suite Dreams?

Failed: facts_correct

```json
{
  "status": "answered",
  "answer": "Koki Mitani",
  "movies": [
    {
      "movie_id": "21122",
      "title": "Suite Dreams",
      "reason": "The movie's director is stated as Koki Mitani in the context."
    }
  ]
}
```

Reference:
```json
{"status": "answered", "answer": "The director is Koki Mitani.", "movies": [{"movie_id": "21122", "title": "Suite Dreams", "reason": "Director: Koki Mitani"}]}
```

### Sample 3: What genres does Suite Dreams belong to?

Failed: facts_correct

```json
{
  "status": "answered",
  "answer": "Comedy, Drama, Romance",
  "movies": [
    {
      "movie_id": "21122",
      "title": "Suite Dreams",
      "reason": "The movie 'Suite Dreams' belongs to the genres Comedy, Drama, and Romance as stated in the context."
    }
  ]
}
```

Reference:
```json
{"status": "answered", "answer": "The movie belongs to these genres: Comedy, Drama, Romance.", "movies": [{"movie_id": "21122", "title": "Suite Dreams", "reason": "Genres: Comedy, Drama, Romance"}]}
```

### Sample 4: Who directed M'Liss?

Failed: facts_correct

```json
{
  "status": "answered",
  "answer": "Marshall Neilan",
  "movies": [
    {
      "movie_id": "300",
      "title": "M'Liss",
      "reason": "The movie's director is given as Marshall Neilan in the overview."
    }
  ]
}
```

Reference:
```json
{"status": "answered", "answer": "The director is Marshall Neilan.", "movies": [{"movie_id": "300", "title": "M'Liss", "reason": "Director: Marshall Neilan"}]}
```

## adapter: failures by question type

{"director": {"n": 99, "all_checks": 99}, "genres": {"n": 99, "all_checks": 99}}

Failure counts: {}

Token-limit hits: 0.

CSV for manual review: /kaggle/working/w2w_english_v1/reports/paired_comparison.csv


## 10. Đóng gói artifacts
Giữ adapter, tokenizer, train arguments/state/history, manifest/hash dữ liệu, model lock, environment/pip freeze, script và toàn bộ báo cáo/predictions. ZIP chứa cả checkpoint optimizer để có thể resume; file base model HF nằm trong cache và **không** đưa vào ZIP. Không merge/quantize GGUF, không triển khai Ollama hay ứng dụng.

Có thể chạy lại cell đóng gói sau giai đoạn baseline hoặc một lần train bị ngắt để lưu trạng thái đang có. Nếu chưa có checkpoint, train sẽ bắt đầu lại; inference vẫn tiếp tục theo mẫu đã lưu.


In [37]:
# Index all stable deliverables; exclude temporary files.
artifact_index={str(p.relative_to(ROOT)):file_hash(p) for p in ROOT.rglob('*')
                if p.is_file() and not p.name.endswith('.tmp') and p.name!='artifact_index.json'}
(ROOT/'artifact_index.json').write_text(json.dumps(artifact_index,indent=2))
archive=shutil.make_archive(str(WORK/('w2w_'+RUN_NAME+'_artifacts')),'zip',root_dir=ROOT)
from IPython.display import FileLink
print('Artifact directory:',ROOT,'\nZIP:',archive)
display(FileLink(archive))


Artifact directory: /kaggle/working/w2w_english_v1 
ZIP: /kaggle/working/w2w_english_v1_artifacts.zip


/kaggle/working/w2w_english_v1_artifacts.zip

In [38]:
from pathlib import Path

p = Path("/kaggle/working/w2w_english_v1_artifacts.zip")
print("Dung lượng:", p.stat().st_size, "bytes")

Dung lượng: 622950435 bytes


## Giới hạn và cách đọc kết quả
- Dữ liệu gốc vốn được sinh từ template catalog; notebook không tạo thêm dữ liệu. Tập test có 99 câu đạo diễn + 99 câu thể loại, **tất cả answered**. Status accuracy cao không chứng minh model biết no_match/insufficient_context.
- Không đo recommendation nhiều phim, xử lý overview, hội thoại/memory, prompt injection, câu hỏi tự nhiên đa dạng hoặc tiếng Việt.
- Reference và context được kiểm tra nhất quán, nhưng không xác minh độ đúng của catalog ngoài đời.
- Fact metric chặt có thể phạt paraphrase đúng. Xem paired CSV trước khi kết luận; không âm thầm sửa bộ chấm sau khi nhìn kết quả để tăng điểm.
- Base ở đây là base model chạy NF4, cùng quantization adapter inference; không phải baseline FP16.
- Một seed train và một test cố định không đủ chứng minh khả năng tổng quát. Không dùng kết quả test này để chọn hyperparameter rồi báo lại như đánh giá chưa từng thấy.
- Cache chỉ phục hồi được bytes còn tồn tại hoặc outputs đã lưu. Không notebook nào đảm bảo phục hồi outputs của runtime đã bị xóa mà chưa lưu.

## Nguồn kỹ thuật chính thức
- [Qwen2.5-3B-Instruct model card và chat template usage](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct)
- [Transformers 4.53.3 Trainer](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/trainer)
- [PEFT quantization / QLoRA](https://huggingface.co/docs/peft/v0.17.0/en/developer_guides/quantization)
- [bitsandbytes 0.48.2 installation và CUDA compatibility](https://huggingface.co/docs/bitsandbytes/v0.48.2/installation)

**Kiểm chứng trước khi bàn giao**: cấu trúc notebook, cú pháp tất cả script/cell, audit 1.996 mẫu thật, gold scoring và các trường hợp output sai được kiểm tra ở môi trường tạo file. Chưa chạy train/inference GPU tại đây; phải chạy mục 4 trên Kaggle để kiểm chứng stack CUDA thực của phiên bạn dùng. Không có số liệu model mới được bịa sẵn.
